# 11 — The five surfaces

Step 10 completes the Registry and proves the extension-locality law: every
capability in this chapter lands through a **registration**, and the kernel
— now at exactly its 1150-line cap, finished — was not edited to make any
of it possible.

The five doors (architecture D3):
**A** ops (`defop`) · **B** overloads/batteries (`intrinsic`, `@overload`,
record methods) · **C** types (`@record`) · **D** backends (`spell`,
`code_for_op`) · **E** the Registry itself (`extend()` layering,
entry-point discovery).

Also in this step: the ch07a **table stakes** graduated — aug-assign,
short-circuit `and`/`or`, chained comparisons, tuples with unpacking, and
subscripts are base-pack registrations now. The next interlude (11a)
records the deltas.

In [1]:
import pdum.dsl  # noqa: F401  — batteries: base dialect + demo backends + the battery pack
from pdum.dsl import jit, no_compile
from pdum.dsl.kernel.registry import DEFAULT


def make(lo, hi):
    @jit()
    def f(x):
        x += 1.0  # aug-assign (was a MissingRule in ch07a)
        a, b = x * 2.0, x + 0.5  # tuple literal + unpack
        a, b = b, a  # the swap idiom
        inside = 0.0 < x < 10.0 and a > b  # chained compare + short-circuit and
        return clamp(a, lo, hi) if inside else smoothstep(0.0, 1.0, x)

    return f


f = make(0.0, 5.0)
print("f(2.0)  =", f(2.0), "  (clamped upper branch)")
print("f(-3.0) =", f(-3.0), " (smoothstep branch — x+1 < 0)")

f(2.0)  = 1.0   (clamped upper branch)
f(-3.0) = 0.0  (smoothstep branch — x+1 < 0)


`and` compiles to `core.if` with the untaken side owned by its branch — the
dominator-placed renderers keep short-circuit REAL on every backend, no new
mechanism. `clamp` and `smoothstep` are **DSL-written batteries**: ordinary
Python registered through `@overload`, inlined at every call site, portable
to every target that exists or ever will.

## Surface B + D: intrinsics and spellings

`sqrt` is the other battery species: an *op* (`math.sqrt`) with a per-target
*spelling*. The rendered source shows the spelling chosen for this target:

In [2]:
@jit()
def hyp(a, b):
    return sqrt(a * a + b * b)


print("hyp(3, 4) =", hyp(3.0, 4.0))
key = next(iter(DEFAULT.specializations._ready))
print(DEFAULT.specializations._ready[key].artifact.__pdum_source__)

hyp(3, 4) = 5.0
import math
from struct import unpack_from as _u
from pdum.dsl.kernel.registry import Out as _Out

def _tdiv(a, b):  # exact trunc division (numeric policy: C semantics)
    q = a // b
    return q + 1 if q < 0 and q * b != a else q

def _tmod(a, b):
    return a - _tdiv(a, b) * b

def kernel(staging, leaves):
    if any(isinstance(x, _Out) for x in leaves):
        raise TypeError("the python backend takes no launcher data (out= is for device targets)")
    v0 = 0.0
    v1 = _u('<d', staging, 16)[0]
    v2 = 1.0
    v3 = v1 + v2
    v4 = v0 < v3
    if v4:
        v5 = 10.0
        v6 = v3 < v5
        v8 = v6
    else:
        v7 = False
        v8 = v7
    v9 = 2.0
    v10 = v3 * v9
    v11 = 0.5
    v12 = v3 + v11
    v13 = (v10, v12)
    v14 = v13[1]
    v15 = v13[0]
    v16 = (v14, v15)
    v17 = v16[0]
    if v8:
        v18 = v16[1]
        v19 = v17 > v18
        v21 = v19
    else:
        v20 = False
        v21 = v20
    if v21:
        v22 = _u('<d', stagin

## The plan's promised demo: `sinh`, live, without restarting

Register the op (A), the call name (B) — and NO spelling. The shared
decomposition fires because the target lacks the native op (the §2.10 gate:
`op not in backend.code_for_op`). Then hand the target a native spelling
and watch the SAME kernel shape compile differently:

In [3]:
import math

from pdum.dsl.kernel.rewrite import Pat
from pdum.dsl.stdlib.surfaces import defop, intrinsic, spell

defop(DEFAULT, "math.sinh", lambda args, attrs, regions: args[0])
intrinsic(DEFAULT, "sinh", "math.sinh")
DEFAULT.decompositions.append(
    (
        "math.sinh",
        (
            Pat("math.sinh"),
            lambda b, m: b.emit(
                "core.div",
                b.emit(
                    "core.sub",
                    b.emit("math.exp", m["root"].args[0]),
                    b.emit("math.exp", b.emit("core.neg", m["root"].args[0])),
                ),
                b.emit("core.const", type=m["root"].type, value=2.0),
            ),
        ),
    )
)


@jit()
def s1(x):
    return sinh(x)


print("decomposed:", s1(0.3), "vs math.sinh:", math.sinh(0.3))
src = list(DEFAULT.specializations._ready.values())[-1].artifact.__pdum_source__
print("  rendered via:", "exp-decomposition" if "math.exp(" in src else "native")

spell(DEFAULT, "demo.simple_shader.python", "math.sinh", "math.sinh({0})")


@jit()
def s2(x):
    return sinh(x)  # same shape, NEW build: the gate now finds the native op


print("native    :", s2(0.3))
src2 = list(DEFAULT.specializations._ready.values())[-1].artifact.__pdum_source__
print("  rendered via:", "native math.sinh" if "math.sinh(" in src2 else "decomposition")

decomposed: 0.30452029344714265 vs math.sinh: 0.3045202934471426
  rendered via: exp-decomposition
native    : 0.3045202934471426
  rendered via: native math.sinh


## Surface C: the `Color` record, end to end

A frozen dataclass becomes a first-class captured value: `typeof` gives a
`Record`, its fields flatten to uniform slots (the ch08 walkers already knew
how), and its **methods inline** like any callee — the ch07a jitclass story,
delivered.

One deliberate detail: `Color` lives in `pdum.dsl.demo.graphics`, NOT the
stdlib. Real color modeling — spaces, RGB vs Lab vs OkLab — is a domain
library's ground, and the stdlib must not squat on it (design doc 090). The
whole point of the five surfaces is that domain vocabulary arrives like any
package: one import wires it in.

In [4]:
from pdum.dsl.demo.graphics import Color  # demo vocabulary: the import IS the installation

c = Color(0.8, 0.4, 0.2)
c = Color(0.8, 0.4, 0.2)


def tint(color, k):
    @jit()
    def f(x):
        return color.luminance() * x + color.scaled(k)[0]

    return f


print("tint =", tint(c, 0.5)(1.0))
key = next(iter(reversed(DEFAULT.specializations._ready)))
rec = DEFAULT.specializations._ready[key]
print("the Color capture became", len(rec.plan.slots) - 0, "uniform slots:")
for s in rec.plan.slots:
    print("  ", s.source, "->", s.dest)

tint = 0.8706
the Color capture became 5 uniform slots:
   LeafPath(root='env', index=0, sub=(0,)) -> PackedDest(offset=0, fmt='<d')
   LeafPath(root='env', index=0, sub=(1,)) -> PackedDest(offset=8, fmt='<d')
   LeafPath(root='env', index=0, sub=(2,)) -> PackedDest(offset=16, fmt='<d')
   LeafPath(root='env', index=1, sub=()) -> PackedDest(offset=24, fmt='<d')
   LeafPath(root='arg', index=0, sub=()) -> PackedDest(offset=32, fmt='<d')


In [5]:
def shade(color, cx, cy, r):
    @jit(kind="simple_shader.compute")
    def disk(i, j):
        x = i / 32.0 - 1.0
        y = j / 32.0 - 1.0
        d = sqrt((x - cx) * (x - cx) + (y - cy) * (y - cy))
        return color.luminance() * (1.0 - smoothstep(r - 0.1, r + 0.1, d))

    return disk


img = shade(Color(0.9, 0.7, 0.3), 0.0, 0.0, 0.6)(out=(64, 64))
shades = " ░▒▓█"
for j in range(0, 64, 4):
    print("".join(shades[min(4, int(img[j * 64 + i] * 6))] for i in range(0, 64, 2)))
c0 = DEFAULT.specializations.compiles
with no_compile():
    shade(Color(0.2, 0.9, 0.5), 0.3, -0.2, 0.4)(out=(64, 64))
print("new Color, new geometry, same types ->", DEFAULT.specializations.compiles - c0, "compiles")
print("(smoothstep INLINED into WGSL; sqrt spelled natively; Color = 3 uniform floats)")

                                
                                
                                
               ░░░              
          ░▒▓▓█████▓▓▒░         
        ░▒█████████████▒░       
       ░▓███████████████▓░      
       ▒█████████████████▒      
      ░▓█████████████████▓░     
       ▒█████████████████▒      
       ░▓███████████████▓░      
        ░▒█████████████▒░       
          ░▒▓▓█████▓▓▒░         
               ░░░              
                                
                                
new Color, new geometry, same types -> 0 compiles
(smoothstep INLINED into WGSL; sqrt spelled natively; Color = 3 uniform floats)


The soft edge is `smoothstep` — the same DSL battery that ran on the CPU —
inlined into the WGSL text. The batteries did not know this backend exists.

## Surface E: layering and discovery

`extend()` gives a child registry with copied registrations and FRESH
caches: a session can add vocabulary without touching the parent — the
stdlib → user → session story:

In [6]:
child = DEFAULT.extend()
defop(child, "math.cbrt", lambda args, attrs, regions: args[0])
intrinsic(child, "cbrt", "math.cbrt")
spell(child, "demo.simple_shader.python", "math.cbrt", "({0} ** (1.0/3.0))")


@jit()
def croot(x):
    return cbrt(x)


print("child :", child.dispatch(croot, (27.0,)))
print("parent knows math.cbrt?", "math.cbrt" in DEFAULT.ops)


class FakeEP:  # what a pip-installed backend distribution provides (080 §4)
    name = "acme-npu"

    def load(self):
        return lambda registry: print("  acme-npu install(registry) called")


print("entry points loaded:", DEFAULT.extend().load_entry_points(entries=[FakeEP()]))

child : 3.0
parent knows math.cbrt? False
  acme-npu install(registry) called
entry points loaded: ['acme-npu']


## The exit gate: battery economics

The numba lesson (architecture risk #4): batteries written IN the language
are portable to every target for free; hand-spelled intrinsics cost one
entry per op per target, forever. The count so far:

In [7]:
dsl_written = [n for n, v in DEFAULT.overloads.items() if not isinstance(v, (str, tuple)) or isinstance(n, tuple)]
intrinsics = sorted({v for v in DEFAULT.overloads.values() if isinstance(v, str)})
spellings = sum(len(b.code_for_op) for b in DEFAULT.backends.values())
print(f"DSL-written batteries (portable to ALL targets, free): {len(dsl_written)}")
print(f"hand-spelled intrinsic ops:                            {len(intrinsics)}")
print(f"spelling entries paid across today's targets:          {spellings}")
print()
print("Every new target pays ~", len(intrinsics), "spellings and inherits", len(dsl_written), "batteries free.")
print("The ratio improves monotonically with both axes — that is the numba 2:1")
print("economics, structural instead of aspirational.")

DSL-written batteries (portable to ALL targets, free): 13
hand-spelled intrinsic ops:                            9
spelling entries paid across today's targets:          27

Every new target pays ~ 9 spellings and inherits 13 batteries free.
The ratio improves monotonically with both axes — that is the numba 2:1
economics, structural instead of aspirational.


## Things to notice

- The kernel is FINISHED: 1150/1150 counted lines, and this entire chapter
  — new syntax, new ops, new types, new spellings, layered registries —
  landed with **zero kernel diffs**. The extension-locality law is now a
  test (`tests/test_surfaces.py`), not a slogan.
- Tuples exist on the GPU only as a fiction: `extract`-of-`tuple` folds away
  via a decomposition GATED on `core.tuple ∉ code_for_op` — the same gate
  that chose sinh's fate. One mechanism, two jobs.
- Record methods are inlined DSL code — `luminance()` became three
  multiplies in both backends' rendered source.

## What we can't do yet

Vec/color RETURNS (a fragment still yields one scalar — vec types land with
the grown-up WGSL backend); `@overload` target-token MRO (deferred until a
battery's BODY must differ per target — decompositions cover today's cases);
`@overload_attribute`; `to_oklab` (wants vec math). Statements (`if`/`for`,
early return) are step 11, where arrays make them worth having. **Next: the
lay-of-the-land delta (11a), then arrays.**